In [ ]:
%%capture

import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ.get("CUDA_VISIBLE_DEVICES")

# print("CUDA_VISIBLE_DEVICES:", os.environ.get("CUDA_VISIBLE_DEVICES"))

In [ ]:
import networkx as nx
from unsloth import FastLanguageModel
from unsloth.chat_templates import get_chat_template
try:
    from ..utils import *
except ImportError:
    import os
    import sys

    current_dir = os.path.dirname(os.path.abspath(__file__))
    project_root = os.path.abspath(os.path.join(current_dir, ".."))
    sys.path.insert(0, project_root)

    from utils import *


class LLMFiller:
    def __init__(self, model_path, max_seq_length=2048, dtype=None, load_in_4bit=True):
       
        self.max_seq_length = max_seq_length
        self.dtype = dtype
        self.load_in_4bit = load_in_4bit
        self.model, self.tokenizer = FastLanguageModel.from_pretrained(
            model_name=model_path,
            max_seq_length=self.max_seq_length,
            dtype=self.dtype,
            load_in_4bit=self.load_in_4bit,
        )

    def llm_fill(self, question):
        FastLanguageModel.for_inference(self.model)  

        messages = [
            {"role": "user", "content": '''Fill in the missing entity names in the JSON structure. This structure is related to activities in a Linux system and may involve various system entities, such as processes, files, or network-related components. Each entity should be named appropriately based on its type and its action on the next entity in the JSON. Please provide realistic and diverse names for these entities.\n{}'''.format(question)},
        ]

        inputs = self.tokenizer.apply_chat_template(
            messages,
            tokenize=True,
            add_generation_prompt=True, 
            return_tensors="pt",
        ).to("cuda")

        outputs = self.model.generate(
            input_ids=inputs,
            max_new_tokens=256,
            use_cache=True,
            temperature=1.5,
            min_p=0.1
        )
        outputs = self.tokenizer.batch_decode(outputs)[0]

        start_marker = "<|start_header_id|>assistant<|end_header_id|>\n\n"
        end_marker = "<|eot_id|>"

        start_index = outputs.find(start_marker)
        end_index = outputs.find(end_marker, start_index)

        if start_index != -1 and end_index != -1:
            assistant_output = outputs[start_index + len(start_marker):end_index].strip()
            return assistant_output
        else:
            return None
        



In [ ]:
datasets = ['optc_h501']
dataset = datasets[0]

model_path = f'lora_models/{dataset}_lora_model'
llm_filler = LLMFiller(model_path)

In [ ]:
import torch
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
print(f"GPU = {gpu_stats.name}. Max memory = {max_memory} GB.")
print(f"{start_gpu_memory} GB of memory reserved.")

In [ ]:


def format_path_for_filling(G, path):
    question = []
    # type_map = {0: "process", 1: "file", 2: "network"}
    
    ok = False

    for i, node_id in enumerate(path):
        node_data = G.nodes[node_id]
        node_label = node_data.get("label", "")
        # if type(node_label) != str:
        #     node_label = type_map.get(node_label, "")
        node_name = node_data.get("name", "[FILL]")
        if node_name == "[FILL]":
            ok = True
        node_action = "null"

        if i + 1 < len(path):
            edge_data = G.get_edge_data(node_id, path[i + 1])
            node_action = edge_data.get("label", "") if edge_data else ""

        node_question = {"type": node_label, "name": node_name, "action": node_action}
        question.append(node_question)

    if ok:
        return json.dumps(question)
    else:
        return None



def update_graph(G, filled_question, path):
    filled_question = filled_question.replace("'", '"')
    filled_question = filled_question.replace("None", 'null')
    filled_question = json.loads(filled_question)
    
    for i, node_id in enumerate(path):
        node_data = G.nodes[node_id]
        if "name" not in node_data:
            node_data["name"] = filled_question[i]["name"]

input_folder_path = f"synGraphs_noname/{dataset}/"
output_folder_path = f"synGraphs_name/{dataset}"
os.makedirs(output_folder_path, exist_ok=True)
n = 30
all_files = os.listdir(input_folder_path)
all_files.sort(key=lambda x: int(os.path.splitext(x)[0].replace("graph", "")))
file_names = all_files[:n]


for file_name in file_names:
        input_file_path = os.path.join(input_folder_path, file_name)


        base_name, _ = os.path.splitext(file_name)
        output_file_name = f"{base_name}.graphml"
        output_file_path = os.path.join(output_folder_path, output_file_name)

        if os.path.exists(output_file_path):
            continue

        G = load_graph(input_file_path)
        all_dfs_paths = get_dfs_paths(G)

        for path in all_dfs_paths:
            question = format_path_for_filling(G, path)

            if question == None:
                continue

            max_attempts = 3
            attempts = 0
            success = False


            while attempts < max_attempts and not success:
                filled_question = llm_filler.llm_fill(question)
                print(filled_question)
                if filled_question:
                    try:
                        update_graph(G, filled_question, path)
                        success = True  
                    except Exception as e:
                        print(f"Error : {filled_question}")
                attempts += 1


        base_name, _ = os.path.splitext(file_name)
        output_file_name = f"{base_name}.graphml"
        output_file_path = os.path.join(output_folder_path, output_file_name)
        save_graph(G, output_file_path)

In [ ]:
used_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
used_memory_for_lora = round(used_memory - start_gpu_memory, 3)
used_percentage = round(used_memory / max_memory * 100, 3)
lora_percentage = round(used_memory_for_lora / max_memory * 100, 3)
print(f"Peak reserved memory = {used_memory} GB.")
print(f"Peak reserved memory for training = {used_memory_for_lora} GB.")
print(f"Peak reserved memory % of max memory = {used_percentage} %.")
print(f"Peak reserved memory for training % of max memory = {lora_percentage} %.")